# Baselines B0 and B1

Simple models that set the score our image models must beat.

| ID | Model | Input |
| --- | --- | --- |
| B0 | Training-set mean for every series | Nothing |
| B1 | Logistic regression / LightGBM | Age, sex, modality |

**Evaluation (same for every model, also B2 and the final model):**
- The shared 5-fold split `splits/folds.csv` (`src.utils.load_folds`).
- For each fold, the model is trained on the other 4 folds and predicts this fold (`src.utils.run_cv`).
  The result is one out-of-fold (OOF) prediction for every series.
- Each fold is scored with the competition metric on its own, then the 5 scores are averaged (`src.utils.cv_report`).
- OOF predictions are saved to `results/oof_<model>.csv` for the final comparison.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils import LABEL_COLS, cv_report, load_folds, run_cv, save_oof

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
folds = load_folds()
results = {}  # model name -> summary scores


def show(name, report):
    """Print the scores of one model and keep its summary for the final table."""
    results[name] = report["summary"]
    print(report["summary"].round(4).to_string())
    print("\nScore per fold:", report["per_fold"].round(4).tolist())
    print("\nScore per modality:")
    print(report["per_modality"].round(4).to_string())

## B0 — training-set mean

For every label, B0 predicts the share of positives in the training folds, the same number for every series.
AUC only looks at the *order* of the predictions, and here every series gets the same value,
so every AUC is exactly 0.5. B0 is a sanity check of the evaluation code: if it scored above 0.5,
labels from the validation fold would be leaking into the predictions.

In [2]:
def b0_fit_predict(train_part, valid_part):
    means = train_part[LABEL_COLS].mean().to_numpy()
    return np.tile(means, (len(valid_part), 1))


oof_b0 = run_cv(train, b0_fit_predict, folds)
show("B0", cv_report(train, oof_b0))
save_oof(oof_b0, "b0")
oof_b0.head(3)

cv_score             0.5
cv_std               0.0
present_auc          0.5
mean_location_auc    0.5

Score per fold: [0.5, 0.5, 0.5, 0.5, 0.5]

Score per modality:
CTA           0.5
MRA           0.5
MRI T1post    0.5
MRI T2        0.5


,SeriesInstanceUID,fold,Left Infraclinoid Internal Carotid Artery,Right Infraclinoid Internal Carotid Artery,Left Supraclinoid Internal Carotid Artery,Right Supraclinoid Internal Carotid Artery,Left Middle Cerebral Artery,Right Middle Cerebral Artery,Anterior Communicating Artery,Left Anterior Cerebral Artery,Right Anterior Cerebral Artery,Left Posterior Communicating Artery,Right Posterior Communicating Artery,Basilar Tip,Other Posterior Circulation,Aneurysm Present
0,1.2.826.0.1.3680043.8.498.10004044428023505108...,1,0.018401,0.022139,0.075906,0.064117,0.049741,0.067855,0.083381,0.010063,0.012938,0.020127,0.022714,0.025014,0.026164,0.428695
1,1.2.826.0.1.3680043.8.498.10004684224894397679...,3,0.017534,0.022995,0.075596,0.064386,0.050877,0.065536,0.084794,0.010635,0.012935,0.020121,0.023570,0.025582,0.025870,0.428284
2,1.2.826.0.1.3680043.8.498.10005158603912009425...,1,0.018401,0.022139,0.075906,0.064117,0.049741,0.067855,0.083381,0.010063,0.012938,0.020127,0.022714,0.025014,0.026164,0.428695


As expected, B0 scores 0.5 on every fold, every label and every modality.
The prediction is the same for all series of a fold (about 0.43 for Aneurysm Present) and differs only slightly between folds.

## Results

In [3]:
pd.DataFrame(results).T.round(4)

,cv_score,cv_std,present_auc,mean_location_auc
B0,0.5,0.0,0.5,0.5
